# J14 v2 — Original TensorFlow/Keras thesis XAI (HAM10000 & BCN20000)

The model architectures in cell 2 are taken **verbatim from G02-A**. This notebook does not use PyTorch.

Requires Kaggle output: `g02-a-v2-ham10000-bcn20000-source-evaluation` (completed G02-A, including prediction chunks and `g02a_verified_test_cohort.csv`) and the original thesis `rotation_ssl_paperlike` trained weights (e.g. `last-part-of-the-previous-one`). Internet ON to download original ISIC 2019 / Hugging Face image archives if not already mounted. GPU recommended. No training.

Each panel: **official original image | the exact HF image input to G02-A | Grad-CAM | Grad-CAM++ (first-gradient formulation) | top-channel Score-CAM**. Fusion XAI traces the actual two SSL encoders and trained Keras fusion head, not a substitute classifier. SimCLR single-row explanation is explicitly the original **validation-selected representative seed**; it is not an attribution of the full multi-seed average. The default fusion attribution is from the SimCLR spatial branch, with an optional rotation-branch panel.

Scientific cautions: G02-A's source identity crosswalk is based on conservative perceptual matching, not a verified original-ID mapping. Explanations are model attribution maps, not lesion segmentation. All reported XAI maps target the model's predicted class. Strict prediction parity with the saved G02-A output is required before drawing maps.

## 1. Environment and verified sample selection

In [1]:
import os,io,json,hashlib,subprocess,zipfile,time,gc,shutil
from pathlib import Path
from collections import OrderedDict
import numpy as np,pandas as pd,tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from PIL import Image,ImageOps
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED=42;np.random.seed(SEED);tf.random.set_seed(SEED)
IMG_SIZE=224;NUM_CLASSES=8
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
SOURCES=['Rotation','SimCLR_Representative','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion_Multiseed']
INPUT=Path('/kaggle/input' if Path('/kaggle/input').is_dir() else '/mnt/data')
WORK=Path('/kaggle/working' if Path('/kaggle/working').is_dir() else '/mnt/data')
TEMP=Path('/kaggle/temp/j14_v2' if Path('/kaggle/temp').is_dir() else '/mnt/data/j14_v2_cache')
OUT=WORK/'j14-original-keras-xai-v2'
for p in [TEMP,OUT/'figures',OUT/'maps',OUT/'reports']:p.mkdir(parents=True,exist_ok=True)
N_PER_SOURCE=int(os.getenv('J14_PER_SOURCE','2'))
CHANNELS=int(os.getenv('J14_SCORECAM_CHANNELS','12'))
BRANCHES=os.getenv('J14_FUSION_BRANCHES','simclr').strip().lower()
MAX_HOURS=float(os.getenv('J14_MAX_HOURS','8'))
START=time.monotonic();BATCH=24
assert BRANCHES in ['simclr','rotation','both'] and 1<=CHANNELS<=32 and N_PER_SOURCE>=1
for gpu in tf.config.list_physical_devices('GPU'):
    try:tf.config.experimental.set_memory_growth(gpu,True)
    except RuntimeError:pass

def req(v,msg):
    if not bool(v):raise RuntimeError(msg)
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()
def write(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(obj,indent=2,default=str,allow_nan=False))
def unique(name,optional=False):
    hits=sorted(p for p in INPUT.rglob(name) if p.is_file())
    if optional and not hits:return None
    req(len(hits)==1,f'Expected one {name}, found {hits[:8]}')
    return hits[0]

cohort_path=unique('g02a_verified_test_cohort.csv')
cohort=pd.read_csv(cohort_path,dtype=str,keep_default_na=False)
req({'hf_id','image_id','source','label','label_id'}.issubset(cohort.columns),'G02-A cohort missing fields')
req(cohort.image_id.is_unique and cohort.hf_id.is_unique,'G02-A accepted IDs are not unique')
cohort['label_id']=cohort.label_id.astype(int)
cohort=cohort.sort_values(['source','image_id']).reset_index(drop=True)
req(set(cohort.source)=={'HAM10000','BCN20000'},'G02-A source labels unexpected')
cohort['original_row']=np.arange(len(cohort))
cohort['choice_hash']=cohort.image_id.map(lambda s:hashlib.sha256(f'J14_v2/{s}'.encode()).hexdigest())
print('Original J1 test cohort:',len(cohort),'images; source counts:',cohort.source.value_counts().to_dict())

status_path=unique('g02a_status.json')
status=json.loads(status_path.read_text())
req(status.get('stage')=='G02-A' and status.get('status')=='COMPLETED','G02-A predictions incomplete')
req(int(status['images'])==len(cohort),'G02-A cohort and prediction count differ')
req(status.get('classes')==CLASSES,'G02-A saved class ordering mismatch')
G02_ROOT=status_path.parent.parent
REF={}
for chunk in status['chunks']:
    p=G02_ROOT/'predictions'/chunk['file']
    req(p.is_file() and sha(p)==chunk['sha256'],f'Missing or modified G02-A probability file: {p}')
    with np.load(p,allow_pickle=False) as z:
        for j,img in enumerate(z['image_id'].astype(str)):
            req(img not in REF,f'Duplicate G02-A probability ID: {img}')
            REF[img]={k:np.asarray(z['prob__'+k][j],dtype=np.float32) for k in status['models']}
req(set(REF)==set(cohort.image_id),'G02-A cohort does not match its probability archive')

Original J1 test cohort: 2239 images; source counts: {'BCN20000': 1240, 'HAM10000': 999}


## 2. Exact original Keras architecture builders from G02-A

In [2]:
# ============================================================
# EXTERNAL CELL E04: INFERENCE-ONLY MODEL ARCHITECTURES
# ============================================================

SIMCLR_DROPOUT = 0.30
ROTATION_FT_DROPOUT = 0.30

FUSIONCONCAT_DROPOUT = 0.40
FUSIONCONCAT_HIDDEN_DIM_1 = 512
FUSIONCONCAT_HIDDEN_DIM_2 = 256
FUSIONCONCAT_WEIGHT_DECAY = 1e-4

FUSIONGATED_V2_DROPOUT = 0.50
FUSIONGATED_V2_WEIGHT_DECAY = 3e-4
FUSIONGATED_V2_BRANCH_DIM = 256
FUSIONGATED_V2_GATE_HIDDEN = 128
FUSIONGATED_V2_HEAD_HIDDEN = 128
FUSIONGATED_V2_TEMPERATURE = 1.5

BESTV3_BRANCH_DIM = 256
BESTV3_DROPOUT_1 = 0.50
BESTV3_DROPOUT_2 = 0.40
BESTV3_WEIGHT_DECAY = 4e-4
BESTV3_GATE_HIDDEN = 128
BESTV3_HEAD_HIDDEN_1 = 256
BESTV3_HEAD_HIDDEN_2 = 128
BESTV3_TEMPERATURE = 1.0

RESIDUAL_FUSION_BRANCH_DIM = 256
RESIDUAL_FUSION_HIDDEN_DIM = 256
RESIDUAL_FUSION_WEIGHT_DECAY = 3e-4
RESIDUAL_FUSION_DROPOUT = 0.40
RESIDUAL_FUSION_MAX_ALPHA = 0.35


def resnet_preprocess_tensor(images):
    images = tf.cast(
        images,
        tf.float32,
    ) * 255.0

    return tf.keras.applications.resnet50.preprocess_input(
        images
    )


def build_resnet50_encoder(
    name,
    training_flag=False,
):
    inputs = keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        name=f"{name}_input",
    )

    backbone = ResNet50(
        include_top=False,
        weights=None,
        input_shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        pooling="avg",
    )

    features = backbone(
        inputs,
        training=training_flag,
    )

    return keras.Model(
        inputs,
        features,
        name=name,
    )


def build_simclr_finetune_model():
    encoder = build_resnet50_encoder(
        "simclr_resnet50_encoder",
        training_flag=False,
    )

    inputs = keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        name="finetune_img",
    )

    x = layers.Lambda(
        resnet_preprocess_tensor,
        name="resnet_preprocess",
    )(inputs)

    x = encoder(
        x,
        training=False,
    )

    x = layers.Dropout(
        SIMCLR_DROPOUT,
        name="finetune_dropout",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="finetune_classifier",
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name="simclr_finetune_model",
    )

    return model, encoder


def build_rotation_finetune_model():
    encoder = build_resnet50_encoder(
        "rotation_ft_resnet50_encoder",
        training_flag=True,
    )
    encoder.trainable = True

    inputs = keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        name="imagenet_rotation_input",
    )

    x = layers.Lambda(
        resnet_preprocess_tensor,
        name="resnet50_preprocess",
    )(inputs)

    # This reproduces the saved architecture.
    x = encoder(
        x,
        training=True
        if encoder.trainable
        else False,
    )

    x = layers.Dropout(
        ROTATION_FT_DROPOUT,
        name="imagenet_rotation_dropout",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="imagenet_rotation_classifier",
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name="imagenet_rotation_finetune_model",
    )

    return model


def l2_normalize_features(
    features,
    epsilon=1e-12,
):
    features = np.asarray(
        features,
        dtype=np.float32,
    )

    denominator = np.linalg.norm(
        features,
        axis=1,
        keepdims=True,
    )

    return (
        features
        / np.clip(
            denominator,
            epsilon,
            None,
        )
    ).astype(np.float32)


def build_fusionconcat_model(
    input_dim,
):
    inputs = keras.Input(
        shape=(input_dim,),
        name="fusionconcat_input",
    )

    x = layers.BatchNormalization(
        name="bn_input"
    )(inputs)

    x = layers.Dense(
        FUSIONCONCAT_HIDDEN_DIM_1,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONCONCAT_WEIGHT_DECAY
            )
        ),
        name="fusionconcat_dense_1",
    )(x)

    x = layers.BatchNormalization(
        name="bn_1"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_1",
    )(x)
    x = layers.Dropout(
        FUSIONCONCAT_DROPOUT,
        name="dropout_1",
    )(x)

    x = layers.Dense(
        FUSIONCONCAT_HIDDEN_DIM_2,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONCONCAT_WEIGHT_DECAY
            )
        ),
        name="fusionconcat_dense_2",
    )(x)

    x = layers.BatchNormalization(
        name="bn_2"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_2",
    )(x)
    x = layers.Dropout(
        FUSIONCONCAT_DROPOUT,
        name="dropout_2",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="fusionconcat_classifier",
    )(x)

    return keras.Model(
        inputs,
        outputs,
        name="fusionconcat_model",
    )


def stable_softmax_temperature(
    values,
    temperature,
):
    return tf.nn.softmax(
        values / temperature,
        axis=-1,
    )


def build_fusiongated_v2_model(
    simclr_dim,
    rotation_dim,
):
    simclr_input = keras.Input(
        shape=(simclr_dim,),
        name="simclr_input",
    )
    rotation_input = keras.Input(
        shape=(rotation_dim,),
        name="rotation_input",
    )

    simclr_norm = layers.LayerNormalization(
        name="ln_simclr_in"
    )(simclr_input)

    rotation_norm = layers.LayerNormalization(
        name="ln_rotation_in"
    )(rotation_input)

    simclr_projection = layers.Dense(
        FUSIONGATED_V2_BRANCH_DIM,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONGATED_V2_WEIGHT_DECAY
            )
        ),
        name="simclr_proj",
    )(simclr_norm)

    simclr_projection = layers.LayerNormalization(
        name="ln_simclr_proj"
    )(simclr_projection)
    simclr_projection = layers.Activation(
        "relu",
        name="relu_simclr_proj",
    )(simclr_projection)

    rotation_projection = layers.Dense(
        FUSIONGATED_V2_BRANCH_DIM,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONGATED_V2_WEIGHT_DECAY
            )
        ),
        name="rotation_proj",
    )(rotation_norm)

    rotation_projection = layers.LayerNormalization(
        name="ln_rotation_proj"
    )(rotation_projection)
    rotation_projection = layers.Activation(
        "relu",
        name="relu_rotation_proj",
    )(rotation_projection)

    gate_input = layers.Concatenate(
        name="gate_input_concat"
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    gate_hidden = layers.Dense(
        FUSIONGATED_V2_GATE_HIDDEN,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONGATED_V2_WEIGHT_DECAY
            )
        ),
        name="gate_dense_1",
    )(gate_input)

    gate_hidden = layers.LayerNormalization(
        name="ln_gate_1"
    )(gate_hidden)
    gate_hidden = layers.Activation(
        "relu",
        name="relu_gate_1",
    )(gate_hidden)
    gate_hidden = layers.Dropout(
        FUSIONGATED_V2_DROPOUT,
        name="dropout_gate_1",
    )(gate_hidden)

    gate_logits = layers.Dense(
        2,
        activation=None,
        bias_initializer=(
            keras.initializers.Constant(
                [0.30, 0.00]
            )
        ),
        name="gate_logits",
    )(gate_hidden)

    gate_weights = layers.Lambda(
        lambda values: (
            stable_softmax_temperature(
                values,
                FUSIONGATED_V2_TEMPERATURE,
            )
        ),
        name="branch_gate_softmax",
    )(gate_logits)

    alpha = layers.Lambda(
        lambda values: values[:, 0:1],
        name="alpha_weight",
    )(gate_weights)

    beta = layers.Lambda(
        lambda values: values[:, 1:2],
        name="beta_weight",
    )(gate_weights)

    weighted_simclr = layers.Multiply(
        name="weighted_simclr"
    )(
        [
            simclr_projection,
            alpha,
        ]
    )

    weighted_rotation = layers.Multiply(
        name="weighted_rotation"
    )(
        [
            rotation_projection,
            beta,
        ]
    )

    weighted_sum = layers.Add(
        name="weighted_sum"
    )(
        [
            weighted_simclr,
            weighted_rotation,
        ]
    )

    fused = layers.Concatenate(
        name="residual_fused_concat"
    )(
        [
            weighted_sum,
            simclr_projection,
            rotation_projection,
        ]
    )

    x = layers.LayerNormalization(
        name="ln_fused"
    )(fused)

    x = layers.Dense(
        FUSIONGATED_V2_HEAD_HIDDEN,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONGATED_V2_WEIGHT_DECAY
            )
        ),
        name="head_dense_1",
    )(x)

    x = layers.LayerNormalization(
        name="ln_head_1"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_head_1",
    )(x)
    x = layers.Dropout(
        FUSIONGATED_V2_DROPOUT,
        name="dropout_head_1",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="classifier",
    )(x)

    return keras.Model(
        [
            simclr_input,
            rotation_input,
        ],
        outputs,
        name="fusiongated_v2_model",
    )


def build_fusiongated_v3_model(
    simclr_dim,
    rotation_dim,
):
    simclr_input = keras.Input(
        shape=(simclr_dim,),
        name="simclr_input",
    )
    rotation_input = keras.Input(
        shape=(rotation_dim,),
        name="rotation_input",
    )

    simclr_norm = layers.LayerNormalization(
        name="ln_simclr_in"
    )(simclr_input)
    rotation_norm = layers.LayerNormalization(
        name="ln_rotation_in"
    )(rotation_input)

    simclr_projection = layers.Dense(
        BESTV3_BRANCH_DIM,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="simclr_proj",
    )(simclr_norm)
    simclr_projection = layers.LayerNormalization(
        name="ln_simclr_proj"
    )(simclr_projection)
    simclr_projection = layers.Activation(
        "relu",
        name="relu_simclr_proj",
    )(simclr_projection)

    rotation_projection = layers.Dense(
        BESTV3_BRANCH_DIM,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="rotation_proj",
    )(rotation_norm)
    rotation_projection = layers.LayerNormalization(
        name="ln_rotation_proj"
    )(rotation_projection)
    rotation_projection = layers.Activation(
        "relu",
        name="relu_rotation_proj",
    )(rotation_projection)

    gate_input = layers.Concatenate(
        name="gate_input_concat"
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    gate_hidden = layers.Dense(
        BESTV3_GATE_HIDDEN,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="gate_dense_1",
    )(gate_input)
    gate_hidden = layers.LayerNormalization(
        name="ln_gate_1"
    )(gate_hidden)
    gate_hidden = layers.Activation(
        "relu",
        name="relu_gate_1",
    )(gate_hidden)
    gate_hidden = layers.Dropout(
        BESTV3_DROPOUT_1,
        name="dropout_gate_1",
    )(gate_hidden)

    gate_logits = layers.Dense(
        2,
        activation=None,
        bias_initializer=(
            keras.initializers.Constant(
                [0.35, 0.00]
            )
        ),
        name="gate_logits",
    )(gate_hidden)

    gate_weights = layers.Lambda(
        lambda values: (
            stable_softmax_temperature(
                values,
                BESTV3_TEMPERATURE,
            )
        ),
        name="branch_gate_softmax",
    )(gate_logits)

    alpha = layers.Lambda(
        lambda values: values[:, 0:1],
        name="alpha_weight",
    )(gate_weights)
    beta = layers.Lambda(
        lambda values: values[:, 1:2],
        name="beta_weight",
    )(gate_weights)

    weighted_simclr = layers.Multiply(
        name="weighted_simclr"
    )(
        [
            simclr_projection,
            alpha,
        ]
    )
    weighted_rotation = layers.Multiply(
        name="weighted_rotation"
    )(
        [
            rotation_projection,
            beta,
        ]
    )
    weighted_sum = layers.Add(
        name="weighted_sum"
    )(
        [
            weighted_simclr,
            weighted_rotation,
        ]
    )

    absolute_difference = layers.Lambda(
        lambda values: tf.abs(
            values[0] - values[1]
        ),
        name="abs_diff",
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    elementwise_product = layers.Multiply(
        name="elem_prod"
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    fused = layers.Concatenate(
        name="final_fused_concat"
    )(
        [
            weighted_sum,
            simclr_projection,
            rotation_projection,
            absolute_difference,
            elementwise_product,
        ]
    )

    x = layers.LayerNormalization(
        name="ln_fused"
    )(fused)

    x = layers.Dense(
        BESTV3_HEAD_HIDDEN_1,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="head_dense_1",
    )(x)
    x = layers.LayerNormalization(
        name="ln_head_1"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_head_1",
    )(x)
    x = layers.Dropout(
        BESTV3_DROPOUT_1,
        name="dropout_head_1",
    )(x)

    x = layers.Dense(
        BESTV3_HEAD_HIDDEN_2,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="head_dense_2",
    )(x)
    x = layers.LayerNormalization(
        name="ln_head_2"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_head_2",
    )(x)
    x = layers.Dropout(
        BESTV3_DROPOUT_2,
        name="dropout_head_2",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="classifier",
    )(x)

    return keras.Model(
        [
            simclr_input,
            rotation_input,
        ],
        outputs,
        name="fusiongated_best_v3_model",
    )


class BoundedResidualLogitFusion(
    layers.Layer
):
    def __init__(
        self,
        max_alpha=0.35,
        initial_alpha=0.08,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.max_alpha = float(
            max_alpha
        )
        ratio = np.clip(
            initial_alpha / max_alpha,
            1e-4,
            1.0 - 1e-4,
        )
        self.initial_logit = float(
            np.log(
                ratio
                / (1.0 - ratio)
            )
        )

    def build(
        self,
        input_shape,
    ):
        self.alpha_logit = (
            self.add_weight(
                name="alpha_logit",
                shape=(),
                initializer=(
                    keras.initializers.Constant(
                        self.initial_logit
                    )
                ),
                trainable=True,
            )
        )
        super().build(
            input_shape
        )

    def call(
        self,
        inputs,
    ):
        base_logits, correction_logits = inputs
        alpha = (
            self.max_alpha
            * tf.sigmoid(
                self.alpha_logit
            )
        )
        return (
            base_logits
            + alpha
            * correction_logits
        )

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "max_alpha": self.max_alpha,
            }
        )
        return config


def build_residual_fusion_model(
    simclr_dim,
    rotation_dim,
):
    simclr_input = keras.Input(
        shape=(simclr_dim,),
        name="simclr_feature_input",
    )
    rotation_input = keras.Input(
        shape=(rotation_dim,),
        name="rotation_feature_input",
    )
    base_logits_input = keras.Input(
        shape=(NUM_CLASSES,),
        name="base_simclr_logits",
    )

    simclr_projection = (
        layers.LayerNormalization(
            name="sim_ln_in"
        )(simclr_input)
    )
    simclr_projection = layers.Dense(
        RESIDUAL_FUSION_BRANCH_DIM,
        kernel_regularizer=(
            keras.regularizers.l2(
                RESIDUAL_FUSION_WEIGHT_DECAY
            )
        ),
        name="sim_projection",
    )(simclr_projection)
    simclr_projection = (
        layers.LayerNormalization(
            name="sim_ln_proj"
        )(simclr_projection)
    )
    simclr_projection = layers.Activation(
        "relu",
        name="sim_relu",
    )(simclr_projection)

    rotation_projection = (
        layers.LayerNormalization(
            name="rot_ln_in"
        )(rotation_input)
    )
    rotation_projection = layers.Dense(
        RESIDUAL_FUSION_BRANCH_DIM,
        kernel_regularizer=(
            keras.regularizers.l2(
                RESIDUAL_FUSION_WEIGHT_DECAY
            )
        ),
        name="rot_projection",
    )(rotation_projection)
    rotation_projection = (
        layers.LayerNormalization(
            name="rot_ln_proj"
        )(rotation_projection)
    )
    rotation_projection = layers.Activation(
        "relu",
        name="rot_relu",
    )(rotation_projection)

    absolute_difference = layers.Lambda(
        lambda values: tf.abs(
            values[0] - values[1]
        ),
        name="abs_difference",
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    product = layers.Multiply(
        name="elementwise_product"
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    fused = layers.Concatenate(
        name="residual_feature_concat"
    )(
        [
            simclr_projection,
            rotation_projection,
            absolute_difference,
            product,
        ]
    )
    fused = layers.LayerNormalization(
        name="fusion_ln"
    )(fused)
    fused = layers.Dense(
        RESIDUAL_FUSION_HIDDEN_DIM,
        kernel_regularizer=(
            keras.regularizers.l2(
                RESIDUAL_FUSION_WEIGHT_DECAY
            )
        ),
        name="correction_dense",
    )(fused)
    fused = layers.LayerNormalization(
        name="correction_ln"
    )(fused)
    fused = layers.Activation(
        "relu",
        name="correction_relu",
    )(fused)
    fused = layers.Dropout(
        RESIDUAL_FUSION_DROPOUT,
        name="correction_dropout",
    )(fused)
    correction_logits = layers.Dense(
        NUM_CLASSES,
        name="correction_logits",
    )(fused)

    final_logits = (
        BoundedResidualLogitFusion(
            max_alpha=(
                RESIDUAL_FUSION_MAX_ALPHA
            ),
            initial_alpha=0.08,
            name=(
                "bounded_residual_"
                "logit_fusion"
            ),
        )(
            [
                base_logits_input,
                correction_logits,
            ]
        )
    )

    output = layers.Softmax(
        dtype="float32",
        name="final_probabilities",
    )(final_logits)

    return keras.Model(
        [
            simclr_input,
            rotation_input,
            base_logits_input,
        ],
        output,
        name=(
            "simclr_preserving_"
            "residual_fusion"
        ),
    )

## 3. Locate frozen original model weights and reconstruct the trained networks

In [3]:
# Kaggle sometimes publishes a notebook's output as a ZIP; extract model outputs only, never the image archives.
def find_roots(base):
    return sorted(set(p.parent.parent for p in base.rglob('rotation_ssl_encoder_best.weights.h5') if p.parent.name=='rotation_ssl'))
roots=find_roots(INPUT)
if not roots:
    stage=TEMP/'legacy_models';stage.mkdir(exist_ok=True)
    for zp in sorted(INPUT.rglob('*.zip')):
        if zp.name in ('data.zip','ISIC_2019_Training_Input.zip'):continue
        try:
            with zipfile.ZipFile(zp) as z:
                names=z.namelist()
                if not any(s.endswith('rotation_ssl/rotation_ssl_encoder_best.weights.h5') for s in names):continue
                req(sum(i.file_size for i in z.infolist())<8*2**30,'Model ZIP unexpectedly large')
                dest=stage/hashlib.sha256(str(zp).encode()).hexdigest()[:12];dest.mkdir(exist_ok=True)
                for item in z.infolist():
                    if item.is_dir():continue
                    rel=Path(item.filename.replace('\\','/'))
                    req(not rel.is_absolute() and '..' not in rel.parts,'Unsafe ZIP member')
                    target=(dest/rel).resolve()
                    req(target.is_relative_to(dest.resolve()),'Unsafe ZIP extraction')
                    target.parent.mkdir(parents=True,exist_ok=True)
                    if target.is_file() and target.stat().st_size==item.file_size:continue
                    with z.open(item) as fin,open(target,'wb') as fout:shutil.copyfileobj(fin,fout,2**20)
        except zipfile.BadZipFile:continue
    roots=find_roots(stage)
req(len(roots)==1,f'Exactly one original thesis checkpoint root required, got: {roots}')
PIPELINE_ROOT=roots[0]
SIMCLR_DIR=PIPELINE_ROOT/'simclr_finetune_multiseed_bacc_v2'
config=json.loads((SIMCLR_DIR/'multiseed_config.json').read_text())
seed_results=pd.read_csv(SIMCLR_DIR/'multiseed_results.csv')
representative=int(seed_results.sort_values(['val_bacc','val_macro_f1','val_accuracy','seed'],ascending=[False,False,False,True]).iloc[0].seed)
req(representative in [int(s) for s in config['seeds']],'Representative SimCLR seed inconsistent')
resdir=PIPELINE_ROOT/'residual_simclr_rotation_fusion_v1'
resconfig=json.loads((resdir/'config.json').read_text())
paths={
 'Rotation':PIPELINE_ROOT/'extended_imagenet_rotation_finetune/imagenet_rotation_finetune_best.weights.h5',
 'RotationEncoder':PIPELINE_ROOT/'rotation_ssl/rotation_ssl_encoder_best.weights.h5',
 'SimCLR':SIMCLR_DIR/f'seed_{representative}/best_by_val_bacc.weights.h5',
 'FusionConcat':PIPELINE_ROOT/'fusion_concat_multiseed_rep_v2/fusionconcat_best.weights.h5',
 'FusionGatedV2':PIPELINE_ROOT/'fusion_gated_v2_multiseed_rep_v2/fusiongated_v2_best.weights.h5',
 'FusionGatedV3':PIPELINE_ROOT/'fusion_gated_v3_multiseed_rep_v2/fusiongated_best_v3.weights.h5',
}
for seed in resconfig['seeds']:
    paths[f'ResidualFusion_{seed}']=resdir/f'seed_{seed}/best_by_val_bacc.weights.h5'
req(all(p.is_file() for p in paths.values()),'Original thesis weights missing: '+str([k for k,p in paths.items() if not p.is_file()]))

simmodel,simencoder=build_simclr_finetune_model();simmodel.load_weights(str(paths['SimCLR']))
rotmodel=build_rotation_finetune_model();rotmodel.load_weights(str(paths['Rotation']))
rotencoder=build_resnet50_encoder('rotation_resnet50_encoder',training_flag=False)
rotencoder.load_weights(str(paths['RotationEncoder']))
heads={'FusionConcat':build_fusionconcat_model(4096),'FusionGatedV2':build_fusiongated_v2_model(2048,2048),
       'FusionGatedV3':build_fusiongated_v3_model(2048,2048)}
for name in heads:heads[name].load_weights(str(paths[name]))
residual=[]
for seed in resconfig['seeds']:
    m=build_residual_fusion_model(2048,2048);m.load_weights(str(paths[f'ResidualFusion_{seed}']))
    residual.append(m)

def make_probe(encoder):
    candidates=[m for m in encoder.layers if isinstance(m,keras.Model) and 'conv5_block3_out' in [x.name for x in m.layers]]
    req(len(candidates)==1,f'{encoder.name}: ResNet50 spatial backbone not identifiable')
    bb=candidates[0]
    return keras.Model(bb.inputs,[bb.get_layer('conv5_block3_out').output,bb.output],name=encoder.name+'_cam_probe')
rotftencoder=rotmodel.get_layer('rotation_ft_resnet50_encoder')
simprobe=make_probe(simencoder);rotprobe=make_probe(rotencoder);rotftprobe=make_probe(rotftencoder)
ROTATION_BN_MODE=True
RUN_FINGERPRINT=hashlib.sha256(json.dumps({k:sha(p) for k,p in sorted(paths.items())},sort_keys=True).encode()).hexdigest()
counts={'Rotation':rotmodel.count_params(),'SimCLR_Representative':simmodel.count_params(),
        'FusionConcat':simencoder.count_params()+rotencoder.count_params()+heads['FusionConcat'].count_params(),
        'FusionGatedV2':simencoder.count_params()+rotencoder.count_params()+heads['FusionGatedV2'].count_params(),
        'FusionGatedV3':simencoder.count_params()+rotencoder.count_params()+heads['FusionGatedV3'].count_params(),
        'ResidualFusion_Multiseed':simencoder.count_params()+rotencoder.count_params()+sum(m.count_params() for m in residual)}
write(OUT/'reports/frozen_model_provenance.json',{'checkpoint_sha256':{k:sha(p) for k,p in paths.items()},'seed':representative,
 'residual_seeds':resconfig['seeds'],'keras_backend':'tensorflow','model_parameters':counts})
print('Loaded genuine TensorFlow checkpoints; representative SimCLR seed:',representative)

Loaded genuine TensorFlow checkpoints; representative SimCLR seed: 62


## 4. Download source images, resolve archive duplicates, and choose batch-compatible samples

In [4]:
def download(url,target,minsize):
    target=Path(target);target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>=minsize:return target
    cmd=['curl','-fL','--retry','8','--retry-delay','5','--connect-timeout','45','-C','-','-o',str(target),url]
    run=subprocess.run(cmd,check=False)
    req(run.returncode==0 and target.is_file() and target.stat().st_size>=minsize,'Image archive download failed: '+url)
    return target
hf_paths=[p for p in INPUT.rglob('data.zip') if p.is_file() and 'isic' in str(p).lower()]
req(len(hf_paths)<=1,f'Multiple input Hugging Face image archives: {hf_paths}')
hf_path=hf_paths[0] if hf_paths else download('https://huggingface.co/datasets/MKZuziak/ISIC_2019_224/resolve/main/data.zip',TEMP/'data.zip',50*10**6)
raw_paths=[p for p in INPUT.rglob('ISIC_2019_Training_Input.zip') if p.is_file()]
req(len(raw_paths)<=1,f'Multiple official ISIC 2019 input archives: {raw_paths}')
raw_path=raw_paths[0] if raw_paths else download('https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip',TEMP/'ISIC_2019_Training_Input.zip',5*10**9)
with zipfile.ZipFile(hf_path) as z:
    all_members={}
    for m in z.infolist():
        if m.filename.lower().endswith(('.jpg','.jpeg','.png')) and not m.is_dir():
            all_members.setdefault(Path(m.filename).stem,[]).append(m.filename)
    member={};fail={}
    for ident in cohort.hf_id:
        ms=all_members.get(ident,[])
        if not ms:fail[ident]='not_in_archive';continue
        if len(ms)>1:
            shas={hashlib.sha256(z.read(m)).hexdigest() for m in ms}
            if len(shas)!=1:fail[ident]='ambiguous_nonidentical';continue
        member[ident]=sorted(ms)[0]
with zipfile.ZipFile(raw_path) as z:
    raw_member={Path(m.filename).stem:m.filename for m in z.infolist() if m.filename.lower().endswith(('.jpg','.jpeg','.png'))}
cohort['available']=cohort.hf_id.isin(member)&cohort.image_id.isin(raw_member)
HF_SHA=sha(hf_path);RAW_SHA=sha(raw_path)
MODEL_IMAGE_FINGERPRINT=hashlib.sha256((RUN_FINGERPRINT+HF_SHA+RAW_SHA+sha(cohort_path)).encode()).hexdigest()
write(OUT/'reports/input_archive_audit.json',{'hf_archive_sha256':HF_SHA,'official_archive_sha256':RAW_SHA,
 'hf_unresolved':fail,'cohort_hf_available':len(member),'cohort_completely_available':int(cohort.available.sum())})

def decode_hf(ids):
    arrays=[]
    with zipfile.ZipFile(hf_path) as z:
        for ident in ids:
            data=z.read(member[ident]);arr=tf.io.decode_image(data,channels=3,expand_animations=False)
            arr=tf.image.resize(tf.cast(arr,tf.float32),(224,224),method='bilinear')/255.
            arrays.append(tf.ensure_shape(arr,(224,224,3)).numpy())
    return tf.convert_to_tensor(np.stack(arrays,dtype=np.float32))

def batch_spec(row_index):
    lo=(row_index//BATCH)*BATCH;hi=min(lo+BATCH,len(cohort))
    return cohort.iloc[lo:hi],row_index-lo

chosen=[];excluded=[]
for src in ['HAM10000','BCN20000']:
    taken=set()
    for row in cohort.loc[cohort.source.eq(src)].sort_values('choice_hash').itertuples():
        if len(taken)>=N_PER_SOURCE:break
        if row.label_id in taken:continue
        segment,j=batch_spec(row.original_row)
        if not bool(segment.available.all()):
            excluded.append({'image_id':row.image_id,'reason':'HF archive ambiguous/missing for original Rotation inference batch'});continue
        chosen.append({'image_id':row.image_id,'hf_id':row.hf_id,'source':row.source,'label':row.label,
            'label_id':row.label_id,'original_row':row.original_row,'batch_position':j})
        taken.add(row.label_id)
req(chosen,'No original source image has a completely identifiable inference batch')
chosen=pd.DataFrame(chosen)
chosen.to_csv(OUT/'reports/selected_images.csv',index=False)
pd.DataFrame(excluded,columns=['image_id','reason']).to_csv(OUT/'reports/skipped_candidates.csv',index=False)
print('Chosen images',len(chosen),'by source',chosen.source.value_counts().to_dict())

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9318M  100 9318M    0     0  60.4M      0  0:02:34  0:02:34 --:--:-- 48.3M


Chosen images 4 by source {'HAM10000': 2, 'BCN20000': 2}


## 5. Exact original forward graph and prediction equivalence against G02-A

In [5]:
def forward(name,x,branch='simclr',rotation_bn_mode=True):
    x=tf.convert_to_tensor(x,dtype=tf.float32)
    pre=resnet_preprocess_tensor(x)
    if name=='Rotation':
        activation,feature=rotftprobe(pre,training=rotation_bn_mode)
        return activation,rotmodel.get_layer('imagenet_rotation_classifier')(feature,training=False)
    act_s,feat_s=simprobe(pre,training=False)
    if name=='SimCLR_Representative':
        return act_s,simmodel.get_layer('finetune_classifier')(feat_s,training=False)
    act_r,feat_r=rotprobe(pre,training=False)
    s=tf.math.l2_normalize(feat_s,axis=1,epsilon=1e-12)
    r=tf.math.l2_normalize(feat_r,axis=1,epsilon=1e-12)
    if name=='FusionConcat':p=heads[name](tf.concat([s,r],axis=-1),training=False)
    elif name in ['FusionGatedV2','FusionGatedV3']:p=heads[name]([s,r],training=False)
    elif name=='ResidualFusion_Multiseed':
        base=simmodel.get_layer('finetune_classifier')(feat_s,training=False)
        base_logits=tf.math.log(tf.clip_by_value(base,1e-7,1.))
        p=tf.reduce_mean(tf.stack([m([s,r,base_logits],training=False) for m in residual],axis=0),axis=0)
    else:raise RuntimeError('Unknown original source: '+name)
    return (act_s if branch=='simclr' else act_r),p

# Compare the reconstructed spatial graph and original saved G02-A predictions on the exact same inputs.
parity=[]
for row in chosen.itertuples():
    segment,j=batch_spec(row.original_row);xb=decode_hf(segment.hf_id.tolist());x=xb[j:j+1]
    direct_rot=np.asarray(rotmodel(xb,training=False),dtype=np.float32)[j]
    choices={mode:np.asarray(forward('Rotation',xb,rotation_bn_mode=mode)[1],dtype=np.float32)[j] for mode in [False,True]}
    best_mode=min(choices,key=lambda k:float(np.max(np.abs(choices[k]-direct_rot))))
    req(float(np.max(abs(choices[best_mode]-direct_rot)))<3e-4,
        f'{row.image_id}: spatial Rotation graph does not reproduce its own Keras model')
    ref=REF[row.image_id]['Rotation']
    parity.append({'image_id':row.image_id,'model':'Rotation','error_vs_G02A':float(np.max(abs(direct_rot-ref))),
        'reconstruction_error':float(np.max(abs(choices[best_mode]-direct_rot))),'rotation_bn_mode':best_mode})
    for name in ['FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion_Multiseed']:
        p=np.asarray(forward(name,x)[1],dtype=np.float32)[0]
        ref=REF[row.image_id][{'ResidualFusion_Multiseed':'ResidualFusion'}.get(name,name)]
        parity.append({'image_id':row.image_id,'model':name,'error_vs_G02A':float(np.max(abs(p-ref))),
            'reconstruction_error':0.,'rotation_bn_mode':None})
    p0=np.asarray(simmodel(x,training=False),dtype=np.float32)[0]
    p1=np.asarray(forward('SimCLR_Representative',x)[1],dtype=np.float32)[0]
    req(np.max(abs(p0-p1))<3e-4,'Representative SimCLR graph mismatch')
parity_df=pd.DataFrame(parity)
parity_df.to_csv(OUT/'reports/prediction_equivalence.csv',index=False)
print(parity_df.groupby('model').error_vs_G02A.agg(['max','mean']).to_string())
req(parity_df.error_vs_G02A.max()<.025,
    'Some reconstructed predictions do not match G02-A (tolerance=.025). See prediction_equivalence.csv; XAI halted.')
rotation_flags=parity_df.loc[parity_df.model.eq('Rotation'),'rotation_bn_mode'].unique().tolist()
req(len(rotation_flags)==1,'Rotation backbone mode inconsistent across selected images')
ROTATION_BN_MODE=bool(rotation_flags[0])
print('Prediction equivalence confirmed; Rotation BN training mode:',ROTATION_BN_MODE)

                                   max          mean
model                                               
FusionConcat              2.056360e-06  1.154840e-06
FusionGatedV2             1.072884e-06  5.215406e-07
FusionGatedV3             8.940697e-07  2.831221e-07
ResidualFusion_Multiseed  2.980232e-07  1.506414e-07
Rotation                  8.347631e-04  2.109318e-04
Prediction equivalence confirmed; Rotation BN training mode: False


## 6. Grad-CAM, Grad-CAM++, and top-channel Score-CAM

In [6]:
def scale_map(t):
    t=tf.image.resize(tf.cast(t,tf.float32)[...,None],(224,224),method='bilinear')
    arr=np.asarray(t[0,:,:,0],dtype=np.float32)
    arr=np.maximum(arr,0);return (arr-arr.min())/(arr.max()-arr.min()+1e-8)

def gradients(name,xb,j,kind,branch):
    with tf.GradientTape() as tape:
        act,p=forward(name,xb,branch=branch,rotation_bn_mode=ROTATION_BN_MODE)
        tape.watch(act)
        target=int(tf.argmax(p[j]).numpy())
        score=tf.math.log(tf.clip_by_value(p[j,target],1e-7,1.))
    g=tape.gradient(score,act)
    req(g is not None,f'{name} {branch}: no spatial gradient; refusing invalid CAM')
    a=tf.cast(act[j:j+1],tf.float32);g=tf.cast(g[j:j+1],tf.float32)
    if kind=='Grad-CAM':
        weights=tf.reduce_mean(g,axis=(1,2),keepdims=True)
    else:
        g2=tf.square(g);g3=g2*g
        alpha=g2/(2.*g2+tf.reduce_sum(a*g3,axis=(1,2),keepdims=True)+1e-8)
        weights=tf.reduce_sum(alpha*tf.nn.relu(g),axis=(1,2),keepdims=True)
    heat=tf.nn.relu(tf.reduce_sum(a*weights,axis=-1))
    return scale_map(heat),target,float(p[j,target].numpy())

def scorecam(name,xb,j,target,branch):
    act,_=forward(name,xb,branch=branch,rotation_bn_mode=ROTATION_BN_MODE)
    a=np.asarray(act[j],dtype=np.float32)
    energy=np.mean(np.square(a),axis=(0,1))
    chans=np.argsort(-energy)[:min(CHANNELS,len(energy))]
    z=tf.image.resize(tf.gather(act[j:j+1],chans,axis=-1),(224,224),method='bilinear').numpy()[0]
    lo=np.min(z,axis=(0,1),keepdims=True);hi=np.max(z,axis=(0,1),keepdims=True)
    masks=np.clip((z-lo)/(hi-lo+1e-8),0,1).astype(np.float32)
    original=np.asarray(xb,dtype=np.float32)
    scores=[]
    for k in range(len(chans)):
        masked=original.copy();masked[j]*=masks[:,:,k,None]
        pk=forward(name,tf.convert_to_tensor(masked),branch=branch,rotation_bn_mode=ROTATION_BN_MODE)[1]
        scores.append(float(pk[j,target].numpy()))
    scores=np.asarray(scores,dtype=np.float64)
    weights=np.exp(scores-scores.max());weights/=weights.sum()
    cam=np.maximum(np.einsum('hwc,c->hw',masks,weights),0.)
    return (cam-cam.min())/(cam.max()-cam.min()+1e-8)

## 7. Render paper-ready, source-specific XAI panels

In [7]:
metadata_path=OUT/'reports/attribution_metadata.csv'
records=pd.read_csv(metadata_path).to_dict('records') if metadata_path.is_file() else []
for row in chosen.itertuples():
    if time.monotonic()-START>MAX_HOURS*3600:
        print('Runtime guard reached; existing maps/figures retained');break
    segment,j=batch_spec(row.original_row)
    xb=decode_hf(segment.hf_id.tolist());x=xb[j:j+1]
    processed=Image.fromarray(np.clip(np.asarray(x)[0]*255,0,255).astype(np.uint8),'RGB')
    with zipfile.ZipFile(raw_path) as z:
        src=z.read(raw_member[row.image_id])
    with Image.open(io.BytesIO(src)) as im:
        im.load();original=ImageOps.exif_transpose(im).convert('RGB')
    branch_list=['simclr'] if BRANCHES=='simclr' else ['rotation'] if BRANCHES=='rotation' else ['simclr','rotation']
    panel=[('Rotation','rotation'),('SimCLR_Representative','simclr')]
    panel += [(name,branch) for name in SOURCES[2:] for branch in branch_list]
    figpath=OUT/'figures'/f'{row.source}__{row.image_id}.png'
    map_paths=[OUT/'maps'/f'{row.image_id}__{name}__{branch}.npz' for name,branch in panel]
    reuse=figpath.is_file() and all(p.is_file() for p in map_paths)
    if reuse:
        for p in map_paths:
            with np.load(p,allow_pickle=False) as z:
                if 'model_fingerprint' not in z or str(z['model_fingerprint'])!=MODEL_IMAGE_FINGERPRINT:reuse=False;break
    if reuse:
        print('REUSE',figpath.name);continue
    fig,axes=plt.subplots(len(panel),5,figsize=(17,max(3*len(panel),11)),squeeze=False)
    for i,(name,branch) in enumerate(panel):
        active=xb if name=='Rotation' else x
        jj=j if name=='Rotation' else 0
        c,pred,conf=gradients(name,active,jj,'Grad-CAM',branch)
        cpp,pred2,_=gradients(name,active,jj,'Grad-CAM++',branch)
        req(pred==pred2,'Predicted target changed between CAM algorithms')
        sc=scorecam(name,active,jj,pred,branch)
        maps={'gradcam':c,'gradcampp':cpp,'scorecam':sc}
        np.savez_compressed(map_paths[i],**maps,target_class=np.asarray(pred),true_class=np.asarray(row.label_id),
            branch=np.asarray(branch),method=np.asarray(name),model_fingerprint=np.asarray(MODEL_IMAGE_FINGERPRINT))
        axes[i,0].imshow(original);axes[i,1].imshow(processed)
        for k,key in enumerate(['gradcam','gradcampp','scorecam'],start=2):
            axes[i,k].imshow(processed);axes[i,k].imshow(maps[key],alpha=.42,cmap='turbo',vmin=0,vmax=1)
        for ax in axes[i]:ax.set_xticks([]);ax.set_yticks([])
        axes[i,0].set_ylabel(f'{name} [{branch}]\nGT:{row.label}  Pred:{CLASSES[pred]} ({conf:.2f})',fontsize=8)
        if i==0:
            for k,title in enumerate(['Official original','Original thesis input','Grad-CAM','Grad-CAM++','Score-CAM (top channels)']):
                axes[i,k].set_title(title,fontsize=11)
        records.append({'image_id':row.image_id,'source':row.source,'model':name,'branch':branch,'true_label':row.label,
            'predicted_label':CLASSES[pred],'confidence':conf,'attribution_target':'predicted','xai_method':'gradcam/gradcampp/approx_top_channel_scorecam'})
    fig.tight_layout();fig.savefig(figpath,dpi=170,bbox_inches='tight');plt.close(fig)
    print('SAVED',figpath.name)
pd.DataFrame(records).drop_duplicates(['image_id','model','branch'],keep='last').to_csv(metadata_path,index=False)
complete=all((OUT/'figures'/f'{row.source}__{row.image_id}.png').is_file() for row in chosen.itertuples())
write(OUT/'reports/j14_status.json',{'stage':'J14','version':2,'status':'COMPLETED' if complete else 'PAUSED',
 'num_selected':len(chosen),'num_figures':len(list((OUT/'figures').glob('*.png'))),
 'source_protocol':'G02-A original J1 test source-identified ISIC 2019 subset',
 'rotation_batch_context':BATCH,'cam_channels':CHANNELS,'fusion_branches':BRANCHES,
 'checkpoint_fingerprint':RUN_FINGERPRINT,'image_and_model_fingerprint':MODEL_IMAGE_FINGERPRINT,'validated_against_G02A':True,
 'simclr_note':'Representative SimCLR seed only; not an attribution of five-seed mean',
 'gradcampp_note':'First-derivative alpha approximation','scorecam_note':'Top-channel approximation'})
print('J14 v2 status:', 'COMPLETED' if complete else 'PAUSED')

SAVED HAM10000__ISIC_0031888.png
SAVED HAM10000__ISIC_0026259.png
SAVED BCN20000__ISIC_0058408.png
SAVED BCN20000__ISIC_0065458.png
J14 v2 status: COMPLETED
